# DBSINet — Usage Example

Physics-informed neural DBSI parameter estimation.  
Protocol-conditioned Deep Sets architecture.

**Pipeline:**
1. Load data and protocol
2. Train DBSINet (or load a pre-trained checkpoint)
3. Run inference → 12 NIfTI parameter maps
4. Compare with pyDBSI output (optional)

In [ ]:
import os
import numpy as np
import nibabel as nib
import torch

from dbsi_toolbox_net import (
    DBSINet,
    Trainer,
    load_checkpoint,
    run_inference,
    save_maps,
    OUTPUT_MAP_NAMES,
)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Paths — edit these

In [ ]:
DWI_PATH   = '/path/to/dwi.nii.gz'
BVAL_PATH  = '/path/to/dwi.bval'
BVEC_PATH  = '/path/to/dwi.bvec'
MASK_PATH  = '/path/to/mask.nii.gz'
CKPT_DIR   = './checkpoints'
OUTPUT_DIR = './dbsinet_results'
SNR        = 30.0   # override with estimate_snr_robust from pyDBSI if available

## 2. Load data

In [ ]:
img    = nib.load(DWI_PATH)
data   = img.get_fdata().astype(np.float32)
affine = img.affine

bvals = np.loadtxt(BVAL_PATH).astype(np.float32)
bvecs = np.loadtxt(BVEC_PATH).astype(np.float32)
if bvecs.shape[0] == 3 and bvecs.shape[1] != 3:
    bvecs = bvecs.T
norms = np.linalg.norm(bvecs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
bvecs /= norms

mask = nib.load(MASK_PATH).get_fdata().astype(bool)

print(f'DWI shape  : {data.shape}')
print(f'Mask voxels: {mask.sum():,}')
print(f'b_max      : {bvals.max():.0f} s/mm²')
print(f'N volumes  : {len(bvals)}')

## 3a. Train from scratch

In [ ]:
model = DBSINet(
    embed_dim      = 256,
    aggregator_dim = 512,
    n_res_blocks   = 4,
    n_phi_layers   = 3,
    dropout        = 0.0,
)
print(model)

trainer = Trainer(
    model                  = model,
    protocols              = [(bvals, bvecs, SNR)],
    n_samples_per_protocol = 100_000,
    batch_size             = 2048,
    lr                     = 3e-4,
    n_epochs               = 100,
    lambda_start           = 0.5,
    n_anneal_epochs        = 20,
    device                 = 'auto',
    output_dir             = CKPT_DIR,
    save_every             = 10,
)
trainer.train()

## 3b. OR load a pre-trained checkpoint

In [ ]:
CKPT_PATH = os.path.join(CKPT_DIR, 'dbsinet_final.pt')
model, ck_meta = load_checkpoint(CKPT_PATH, device='auto')
print(model)
print(f'Trained for {ck_meta["epoch"]} epochs')

## 4. Inference

In [ ]:
results = run_inference(
    model           = model,
    data            = data,
    bvals           = bvals,
    bvecs           = bvecs,
    mask            = mask,
    fiber_threshold = 0.15,
    batch_size      = 4096,
    device          = 'auto',
)
print(f'Output shape: {results.shape}')   # (X, Y, Z, 12)

## 5. Save NIfTI maps

In [ ]:
save_maps(results, affine, OUTPUT_DIR)
print(f'Maps saved to {OUTPUT_DIR}')
print('Channels:')
for i, name in enumerate(OUTPUT_MAP_NAMES):
    print(f'  {i:2d} : {name}')

## 6. Quick summary statistics

In [ ]:
for ch, name in enumerate(OUTPUT_MAP_NAMES):
    if name.endswith('_NaN'):
        continue
    vol   = results[..., ch][mask]
    valid = vol[~np.isnan(vol)]
    if len(valid) == 0:
        continue
    print(f'  {name:<30}  mean={valid.mean():.4f}  std={valid.std():.4f}  '
          f'valid={len(valid):,}/{mask.sum():,}')